# Pipeline de Datos — CRM + Billing + Universidad

## 1. Objetivo del proyecto

Construir, de extremo a extremo, un pipeline de datos que transforme los CSV crudos
de tres sistemas de origen distintos (CRM, Billing y Universidad), que en conjunto
describen el mismo negocio (una institución que ofrece cursos y factura
suscripciones), en información confiable para el negocio, siguiendo una
arquitectura medallion (Bronze a Silver a Gold) sobre PostgreSQL, con Python/SQL
como motor de transformación y Jupyter como espacio de discovery y validación.

## 2. Descripción de las fuentes de datos

Tres dominios, 18 archivos CSV, cargados sin transformar en `bronze.*` por
`src/ingest/load_raw_bronze.py` (fase de ingesta). El esquema completo (columnas y
filas esperadas) está en `manifest.json`.

In [1]:
import json
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text


def find_project_root(current_dir: Path) -> Path:
    for parent in [current_dir] + list(current_dir.parents):
        if (parent / "manifest.json").exists():
            return parent
    raise FileNotFoundError("No se encontró el archivo manifest.json en ningún directorio padre.")

PROJECT_ROOT = find_project_root(Path.cwd())
manifest = json.loads((PROJECT_ROOT / "manifest.json").read_text())
rows = []

for domain, tables in manifest["domains"].items():
    for table, meta in tables.items():
        rows.append({
            "dominio": domain, 
            "tabla": table, 
            "filas_esperadas": meta["rows"], 
            "columnas": len(meta["cols"])
        })

df_manifest = pd.DataFrame(rows)
df_manifest

,dominio,tabla,filas_esperadas,columnas
0,university,semesters,8,6
1,university,professors,200,6
2,university,students,5000,7
3,university,courses,300,6
4,university,enrollments,25000,6
5,university,grades,60000,6
6,billing,customers,10000,8
7,billing,products,200,6
8,billing,subscriptions,15000,6
9,billing,invoices,50000,7


In [2]:
user = os.environ.get("WAREHOUSE_DB_USER", "postgres")
password = os.environ.get("WAREHOUSE_DB_PASSWORD", "postgres")
db = os.environ.get("WAREHOUSE_DB_NAME", "warehouse")
host = os.environ.get("WAREHOUSE_DB_HOST", "warehouse-db")
port = os.environ.get("WAREHOUSE_DB_PORT", "5432")

In [3]:
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}")

In [4]:
#Comparación de filas cargadas en bronze con manifest
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'bronze'
ORDER BY table_name;
"""
bronze_tables = pd.read_sql(query, engine)["table_name"].tolist()

counts = []
for t in bronze_tables:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM bronze.{t}", engine)["n"].iloc[0]
    counts.append({"tabla_bronze": t, "filas": n})

df_bronze_counts = pd.DataFrame(counts)
df_bronze_counts

,tabla_bronze,filas
0,billing_customers,10000
1,billing_invoice_items,150000
2,billing_invoices,50000
3,billing_payments,80000
4,billing_products,200
5,billing_subscriptions,15000
6,crm_accounts,5000
7,crm_activities,20000
8,crm_contacts,15000
9,crm_leads,2000


## 3. Data Profiling

El perfilado exhaustivo (nulos, duplicados, cardinalidades, tipos, distribuciones)
por tabla se hizo en `notebooks/discovery_university.ipynb`,
`notebooks/discovery_billing.ipynb` y `notebooks/discovery_crm.ipynb`, y sus
hallazgos están consolidados en `docs/decisiones.md` (secciones 1 y 2). Aquí se
verifican en vivo los números clave que sustentan las decisiones de limpieza de la
sección 4.

In [5]:
# Nulos: solo 2 columnas presentan nulos en todo el dataset (por diseño, no por defecto)
query = """
SELECT
  COUNT(*) AS total_activities,
  COUNT(*) FILTER (WHERE contact_id IS NULL) AS contact_id_nulo,
  COUNT(*) FILTER (WHERE opportunity_id IS NULL) AS opportunity_id_nulo
FROM bronze.crm_activities;
"""
pd.read_sql(query, engine)

,total_activities,contact_id_nulo,opportunity_id_nulo
0,20000,5976,9985


In [6]:
# Duplicados por (enrollment_id, assessment) en university_grades
query = """
SELECT n, COUNT(*) AS n_grupos
FROM (
  SELECT enrollment_id, assessment, COUNT(*) AS n
  FROM bronze.university_grades
  GROUP BY enrollment_id, assessment
) t
GROUP BY n
ORDER BY n;
"""
pd.read_sql(query, engine)

,n,n_grupos
0,1,37133
1,2,8954
2,3,1417
3,4,158
4,5,14
5,6,1


## 4. Limpieza y transformaciones realizadas (Silver)

Política general (justificada en `docs/decisiones.md` §3): **no se elimina ni se
"corrige" ninguna fila** por inconsistencia cronológica o de negocio — se tipa,
se estandariza, y se marca con columnas `_dq_*`. Deduplicación solo donde la
evidencia confirma redundancia real de la misma entidad (`university_grades`).

La construcción corre en SQL (`sql/silver/silver_university.sql`,
`silver_billing.sql`, `silver_crm.sql`), no en pandas — es lo que exige el stack del
proyecto ("SQL: modelado y transformaciones de negocio") y mantiene consistencia con
cómo se implementó Bronze (`src/ingest/load_raw_bronze.py`). El notebook solo
ejecuta y valida, no transforma.

In [7]:
import subprocess

result = subprocess.run(
    ["python3", str(PROJECT_ROOT / "src" / "transform" / "build_silver.py")],
    capture_output=True,
    text=True,
    env=os.environ,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0



SILVER BUILD
Ejecutando: silver_university.sql
Ejecutando: silver_billing.sql
Ejecutando: silver_crm.sql


Conteo de filas por tabla silver:
  silver.billing_customers                10,000
  silver.billing_invoice_items           150,000
  silver.billing_invoices                 50,000
  silver.billing_payments                 80,000
  silver.billing_products                    200
  silver.billing_subscriptions            15,000
  silver.crm_accounts                      5,000
  silver.crm_activities                   20,000
  silver.crm_contacts                     15,000
  silver.crm_leads                         2,000
  silver.crm_opportunities                 3,000
  silver.crm_opportunity_contacts          6,000
  silver.university_courses                  300
  silver.university_enrollments           25,000
  silver.university_grades                47,677
  silver.university_professors               200
  silver.university_semesters                  8
  silver.university_stud

In [8]:
# Conteo de filas y coincidencia 1:1, excepto university_grades (deduplicada)
query = """
SELECT table_name FROM information_schema.tables WHERE table_schema = 'silver' ORDER BY table_name;
"""
silver_tables = pd.read_sql(query, engine)["table_name"].tolist()

comparison = []
for t in silver_tables:
    b = pd.read_sql(f"SELECT COUNT(*) AS n FROM bronze.{t}", engine)["n"].iloc[0]
    s = pd.read_sql(f"SELECT COUNT(*) AS n FROM silver.{t}", engine)["n"].iloc[0]
    comparison.append({"tabla": t, "filas_bronze": b, "filas_silver": s, "diferencia": b - s})

df_comparison = pd.DataFrame(comparison)
df_comparison

,tabla,filas_bronze,filas_silver,diferencia
0,billing_customers,10000,10000,0
1,billing_invoice_items,150000,150000,0
2,billing_invoices,50000,50000,0
3,billing_payments,80000,80000,0
4,billing_products,200,200,0
5,billing_subscriptions,15000,15000,0
6,crm_accounts,5000,5000,0
7,crm_activities,20000,20000,0
8,crm_contacts,15000,15000,0
9,crm_leads,2000,2000,0


## 5. Análisis de relaciones entre dominios y justificación

Verificación en vivo de los tres hallazgos de relación documentados en
`docs/decisiones.md` §1.

In [9]:
# university <-> billing: billing.customers.external_ref vs university.students.student_id
query = """
SELECT
  COUNT(*) AS con_external_ref,
  COUNT(s.student_id) AS con_match_en_university,
  COUNT(*) FILTER (WHERE s.country IS NOT NULL AND s.country <> c.country) AS pais_distinto
FROM silver.billing_customers c
LEFT JOIN silver.university_students s ON s.student_id = c.external_ref
WHERE c.external_ref IS NOT NULL;
"""
pd.read_sql(query, engine)

,con_external_ref,con_match_en_university,pais_distinto
0,5000,5000,3928


In [10]:
# crm <-> billing: no hay join explotable por email
query = """
SELECT
  COUNT(*) AS total_contacts,
  COUNT(c.customer_id) AS con_match_en_billing
FROM silver.crm_contacts ct
LEFT JOIN silver.billing_customers c ON c.email = ct.email;
"""
pd.read_sql(query, engine)

,total_contacts,con_match_en_billing
0,15000,1


In [11]:
# crm.leads: tabla aislada, sin conversion reconstruible hacia contacts
query = """
SELECT
  COUNT(*) AS total_leads,
  COUNT(ct.contact_id) AS con_match_en_contacts
FROM silver.crm_leads l
LEFT JOIN silver.crm_contacts ct ON ct.email = l.email;
"""
pd.read_sql(query, engine)

,total_leads,con_match_en_contacts
0,2000,0


## 6. Implementación del pipeline Bronze → Silver: validación de reglas de calidad

Cada columna `_dq_*` marca, sin eliminar la fila, una violación de una regla de
negocio detectada en Bronze (detalle y justificación por tabla en
`docs/decisiones.md` §2.5, §2.6 y §4). Se valida aquí que los conteos coinciden con
lo documentado.

In [12]:
dq_checks = [
    ("silver.university_courses", "_dq_professor_dept_mismatch"),
    ("silver.university_students", "_dq_enrolled_after_first_course"),
    ("silver.university_enrollments", "_dq_outside_semester_range"),
    ("silver.university_grades", "_dq_graded_before_enrollment"),
    ("silver.university_grades", "_dq_graded_after_semester_end"),
    ("silver.billing_customers", "_dq_country_mismatch_university"),
    ("silver.billing_subscriptions", "_dq_start_after_end"),
    ("silver.billing_invoices", "_dq_total_mismatch_items"),
    ("silver.crm_contacts", "_dq_created_before_account"),
    ("silver.crm_opportunities", "_dq_close_before_created"),
    ("silver.crm_activities", "_dq_occurred_before_contact"),
    ("silver.crm_activities", "_dq_occurred_before_opportunity"),
]

rows = []
for table, flag in dq_checks:
    query = f"""
        SELECT COUNT(*) FILTER (WHERE {flag}) AS n_marcadas, COUNT(*) AS total
        FROM {table};
    """
    r = pd.read_sql(query, engine).iloc[0]
    rows.append({
        "tabla": table,
        "flag": flag,
        "filas_marcadas": int(r["n_marcadas"]),
        "total_filas": int(r["total"]),
        "pct": round(100 * r["n_marcadas"] / r["total"], 1),
    })

df_dq = pd.DataFrame(rows)
df_dq

,tabla,flag,filas_marcadas,total_filas,pct
0,silver.university_courses,_dq_professor_dept_mismatch,264,300,88.0
1,silver.university_students,_dq_enrolled_after_first_course,1851,5000,37.0
2,silver.university_enrollments,_dq_outside_semester_range,22729,25000,90.9
3,silver.university_grades,_dq_graded_before_enrollment,21386,47677,44.9
4,silver.university_grades,_dq_graded_after_semester_end,23141,47677,48.5
5,silver.billing_customers,_dq_country_mismatch_university,3928,10000,39.3
6,silver.billing_subscriptions,_dq_start_after_end,789,15000,5.3
7,silver.billing_invoices,_dq_total_mismatch_items,49999,50000,100.0
8,silver.crm_contacts,_dq_created_before_account,7456,15000,49.7
9,silver.crm_opportunities,_dq_close_before_created,1029,3000,34.3


## 7. Modelado Gold: construcción y validación

Implementado en `sql/gold/gold_00_dim_fecha.sql`, `gold_university.sql`,
`gold_billing.sql`, `gold_crm.sql` y `gold_kpis.sql`, ejecutados por
`src/transform/build_gold.py`. Diseño completo (constelación de hechos,
grano de cada fact, dimensión conformada `dim_fecha`, KPIs de negocio)
justificado en `docs/decisiones.md` secciones 5 y 6.

In [13]:
result = subprocess.run(
    ["python3", str(PROJECT_ROOT / "src" / "transform" / "build_gold.py")],
    capture_output=True,
    text=True,
    env=os.environ,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0

Ejecutando: gold_billing.sql
Ejecutando: gold_crm.sql
Ejecutando: gold_kpis.sql


Conteo de filas por tabla gold:
  gold.dim_cliente                      10,000
  gold.dim_contacto                     15,000
  gold.dim_cuenta                        5,000
  gold.dim_curso                           300
  gold.dim_estudiante                    5,000
  gold.dim_fecha                         7,670
  gold.dim_producto                        200
  gold.dim_semestre                          8
  gold.dim_tipo_evaluacion                   5
  gold.fact_actividades                 20,000
  gold.fact_facturas                    50,000
  gold.fact_inscripciones               25,000
  gold.fact_notas                       47,677
  gold.fact_oportunidades                3,000
  gold.fact_pagos                       80,000
  gold.fact_suscripciones               15,000
  gold.kpi_conversion_leads                  5
  gold.kpi_engagement_cuenta             5,000
  gold.kpi_estudiantes_en_riesgo        

In [14]:
# Inventario de tablas gold: dimensiones, hechos y KPIs
query = "SELECT table_name FROM information_schema.tables WHERE table_schema = 'gold' ORDER BY table_name;"
gold_tables = pd.read_sql(query, engine)["table_name"].tolist()

rows = []
for t in gold_tables:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM gold.{t}", engine)["n"].iloc[0]
    tipo = "dim" if t.startswith("dim_") else ("kpi" if t.startswith("kpi_") else ("mart" if t.startswith("mart_") else "fact"))
    rows.append({"tabla": t, "tipo": tipo, "filas": n})

df_gold = pd.DataFrame(rows).sort_values(["tipo", "tabla"])
df_gold

,tabla,tipo,filas
0,dim_cliente,dim,10000
1,dim_contacto,dim,15000
2,dim_cuenta,dim,5000
3,dim_curso,dim,300
4,dim_estudiante,dim,5000
5,dim_fecha,dim,7670
6,dim_producto,dim,200
7,dim_semestre,dim,8
8,dim_tipo_evaluacion,dim,5
9,fact_actividades,fact,20000


In [15]:
# Reconciliacion silver -> gold (mapeo 1:1, ver docs/decisiones.md SS5.6 para las 3 tablas fuera de alcance)
mapeo = {
    "university_semesters": "dim_semestre",
    "university_courses": "dim_curso",
    "university_students": "dim_estudiante",
    "university_enrollments": "fact_inscripciones",
    "university_grades": "fact_notas",
    "billing_customers": "dim_cliente",
    "billing_products": "dim_producto",
    "billing_subscriptions": "fact_suscripciones",
    "billing_invoices": "fact_facturas",
    "billing_payments": "fact_pagos",
    "crm_accounts": "dim_cuenta",
    "crm_contacts": "dim_contacto",
    "crm_opportunities": "fact_oportunidades",
    "crm_activities": "fact_actividades",
    "crm_leads": "mart_leads",
}

rows = []
for silver_t, gold_t in mapeo.items():
    s = pd.read_sql(f"SELECT COUNT(*) AS n FROM silver.{silver_t}", engine)["n"].iloc[0]
    g = pd.read_sql(f"SELECT COUNT(*) AS n FROM gold.{gold_t}", engine)["n"].iloc[0]
    rows.append({"silver": silver_t, "gold": gold_t, "filas_silver": s, "filas_gold": g, "coincide": s == g})

df_reconciliacion = pd.DataFrame(rows)
assert df_reconciliacion["coincide"].all()
df_reconciliacion

,silver,gold,filas_silver,filas_gold,coincide
0,university_semesters,dim_semestre,8,8,True
1,university_courses,dim_curso,300,300,True
2,university_students,dim_estudiante,5000,5000,True
3,university_enrollments,fact_inscripciones,25000,25000,True
4,university_grades,fact_notas,47677,47677,True
5,billing_customers,dim_cliente,10000,10000,True
6,billing_products,dim_producto,200,200,True
7,billing_subscriptions,fact_suscripciones,15000,15000,True
8,billing_invoices,fact_facturas,50000,50000,True
9,billing_payments,fact_pagos,80000,80000,True


## 8. Exploración de la capa Gold: consultas de negocio

Consultas directas sobre las estrellas y los KPIs de Gold — no repiten el
perfilado de calidad (eso ya se hizo sobre Bronze/Silver en las secciones
3-6). Acá la pregunta es de negocio: ¿qué le importa a alguien que dirige
la institución?

### 8.1 Academia: ¿dónde se concentra la reprobación?

In [16]:
# Los 5 cursos con peor tasa de aprobacion
query = """
SELECT codigo, nombre, inscripciones_finalizadas, aprobadas, reprobadas, tasa_aprobacion
FROM gold.kpi_tasa_aprobacion_curso
ORDER BY tasa_aprobacion ASC
LIMIT 5;
"""
pd.read_sql(query, engine)

,codigo,nombre,inscripciones_finalizadas,aprobadas,reprobadas,tasa_aprobacion
0,C-00095,Course 00095,51,34,17,0.6667
1,C-00083,Course 00083,72,51,21,0.7083
2,C-00101,Course 00101,64,47,17,0.7344
3,C-00078,Course 00078,49,36,13,0.7347
4,C-00232,Course 00232,50,37,13,0.7400


In [17]:
# Tasa de aprobacion por semestre: ¿mejora, empeora o se mantiene estable?
query = """
SELECT anio, periodo, inscripciones_finalizadas, aprobadas, reprobadas, tasa_aprobacion
FROM gold.kpi_tasa_aprobacion_semestre
ORDER BY anio, periodo;
"""
pd.read_sql(query, engine)

,anio,periodo,inscripciones_finalizadas,aprobadas,reprobadas,tasa_aprobacion
0,2022,1,2191,1895,296,0.8649
1,2022,2,2259,1909,350,0.8451
2,2023,1,2164,1866,298,0.8623
3,2023,2,2118,1799,319,0.8494
4,2024,1,2140,1795,345,0.8388
5,2024,2,2215,1916,299,0.8650
6,2025,1,2100,1785,315,0.8500
7,2025,2,2275,1966,309,0.8642


In [18]:
# Estudiantes en riesgo (al menos 1 curso reprobado) sobre el total
en_riesgo = pd.read_sql("SELECT COUNT(*) AS n FROM gold.kpi_estudiantes_en_riesgo", engine)["n"].iloc[0]
total_estudiantes = pd.read_sql("SELECT COUNT(*) AS n FROM gold.dim_estudiante", engine)["n"].iloc[0]
print(f"Estudiantes en riesgo: {en_riesgo:,} de {total_estudiantes:,} ({en_riesgo/total_estudiantes:.1%})")

Estudiantes en riesgo: 2,005 de 5,000 (40.1%)


### 8.2 Billing: ¿cuánto se factura, cuánto se cobra, y qué tan sano está el ingreso recurrente?

In [19]:
# Brecha facturado vs cobrado, acumulada sobre todo el historico
query = "SELECT SUM(total_facturado) AS total_facturado, SUM(total_cobrado) AS total_cobrado FROM gold.kpi_facturacion_mensual;"
df_brecha = pd.read_sql(query, engine)
df_brecha["brecha"] = df_brecha["total_facturado"] - df_brecha["total_cobrado"]
df_brecha["brecha_pct"] = df_brecha["brecha"] / df_brecha["total_facturado"]
df_brecha

,total_facturado,total_cobrado,brecha,brecha_pct
0,6791850.14,6392810.36,399039.78,0.058753


In [20]:
# Suscripciones activas con fecha de fin ya vencida (posible fuga de ingreso recurrente sin dar de baja)
query = """
SELECT COUNT(*) AS vencidas, ROUND(AVG(dias_vencida), 1) AS dias_promedio_vencida
FROM gold.kpi_suscripciones_vencidas;
"""
vencidas = pd.read_sql(query, engine)
activas = pd.read_sql("SELECT COUNT(*) AS n FROM gold.fact_suscripciones WHERE status = 'active'", engine)["n"].iloc[0]
vencidas["suscripciones_activas_total"] = activas
vencidas["pct_activas_vencidas"] = vencidas["vencidas"] / activas
vencidas

,vencidas,dias_promedio_vencida,suscripciones_activas_total,pct_activas_vencidas
0,7160,465.6,11272,0.635202


### 8.3 CRM: ¿en qué etapa se traba el pipeline, y qué canal de leads conviene priorizar?

In [21]:
pd.read_sql("SELECT * FROM gold.kpi_pipeline_oportunidades ORDER BY etapa;", engine)

,etapa,num_oportunidades,monto_total,monto_promedio
0,lost,303,11940870.10,39408.81
1,negotiation,420,15370881.80,36597.34
2,proposal,569,22222895.05,39056.05
3,prospect,621,22829672.15,36762.76
4,qualification,611,23040966.86,37710.26
5,won,476,18360511.13,38572.50


In [22]:
pd.read_sql("SELECT * FROM gold.kpi_tasa_cierre_oportunidades;", engine)

,ganadas,perdidas,tasa_cierre_ganado
0,476,303,0.611


In [23]:
pd.read_sql("SELECT * FROM gold.kpi_conversion_leads ORDER BY tasa_conversion DESC;", engine)

,origen,total_leads,convertidos,perdidos,tasa_conversion,puntaje_promedio
0,cold_call,204,30,22,0.1471,48.94
1,referral,402,46,50,0.1144,49.40
2,event,302,32,39,0.1060,50.97
3,ads,282,28,40,0.0993,50.06
4,web,810,69,130,0.0852,50.42


### 8.4 Cruce university ↔ billing: ¿el rendimiento académico se relaciona con cuánto factura un estudiante-cliente?

In [24]:
# Se segmenta por tasa de aprobacion individual y se compara la facturacion promedio de cada segmento
query = """
SELECT
  CASE
    WHEN num_cursos_inscritos = 0 THEN 'sin_cursos'
    WHEN num_cursos_aprobados::NUMERIC / num_cursos_inscritos >= 0.8 THEN 'alto_rendimiento'
    WHEN num_cursos_aprobados::NUMERIC / num_cursos_inscritos >= 0.5 THEN 'rendimiento_medio'
    ELSE 'bajo_rendimiento'
  END AS segmento_academico,
  COUNT(*) AS num_estudiantes,
  ROUND(AVG(total_facturado), 2) AS facturado_promedio
FROM gold.kpi_facturacion_estudiantes
GROUP BY 1
ORDER BY 3 DESC;
"""
pd.read_sql(query, engine)

,segmento_academico,num_estudiantes,facturado_promedio
0,alto_rendimiento,1092,687.95
1,bajo_rendimiento,1269,681.14
2,sin_cursos,38,680.88
3,rendimiento_medio,2601,671.10


In [25]:
# Cuentas CRM: ¿las que tienen una oportunidad abierta reciben mas actividades de venta que las que no tienen ninguna?
query = """
SELECT
  (num_oportunidades > 0) AS tiene_oportunidad,
  COUNT(*) AS num_cuentas,
  ROUND(AVG(num_actividades), 2) AS actividades_promedio,
  ROUND(AVG(monto_pipeline), 2) AS monto_pipeline_promedio
FROM gold.kpi_engagement_cuenta
GROUP BY 1;
"""
pd.read_sql(query, engine)

,tiene_oportunidad,num_cuentas,actividades_promedio,monto_pipeline_promedio
0,False,2738,2.85,0.00
1,True,2262,4.08,50294.34


## 9. Generación de insights: hallazgos concretos y accionables

Basado únicamente en las consultas de la sección 8 (no en la calidad de
datos de Bronze/Silver, que ya se cubrió en 3-6). Cada punto liga un
número verificado a una acción de negocio concreta.

### Academia

- **La reprobación no está distribuida parejo entre cursos**: el curso con
  peor desempeño (`C-00095`) tiene 66,7% de aprobación contra ~85% de
  promedio general — una brecha de casi 20 puntos. Antes de asumir que es
  un problema transversal de la institución, vale la pena revisar
  puntualmente el syllabus/dificultad de los 5-10 cursos peor ubicados en
  `kpi_tasa_aprobacion_curso`, no lanzar una iniciativa genérica para todos
  los cursos.
- **La tasa de aprobación es estable en el tiempo** (84,0%-86,5% en los 8
  semestres, sin tendencia clara de mejora ni deterioro): no es una crisis
  reciente ni un problema que se esté resolviendo solo — es estructural,
  y probablemente tenga más que ver con los cursos puntuales del punto
  anterior que con una tendencia general.
- **40,1% de los estudiantes (2.005 de 5.000) reprobó al menos un curso** —
  proporción alta. A diferencia del punto anterior (concentrado en pocos
  cursos), este número es transversal: sugiere que además de revisar
  cursos específicos, conviene un programa de apoyo académico temprano
  (tutorías, alertas) dirigido a estudiantes con el primer reprobado, antes
  de que acumulen más.

### Billing

- **Brecha histórica facturado vs. cobrado: \$399.039,78 sobre
  \$6.791.850,14 facturados (5,9%)** — no es catastrófico, pero tampoco
  trivial: casi 1 de cada 17 pesos facturados no tiene un pago asociado
  registrado. Ya se documentó en `docs/decisiones.md` §2.5 que
  `invoices.total` no reconcilia con sus líneas ni pagos a nivel de fila;
  este KPI cuantifica el impacto agregado mensual — es el número a llevarle
  a Finanzas para decidir si amerita una auditoría del proceso de cobranza.
- **63,5% de las suscripciones marcadas `active` (7.160 de 11.272) tienen
  `end_date` ya vencido, en promedio hace 465 días (más de 15 meses)** —
  el estado `active` no es un indicador confiable de si el cliente sigue
  activo de verdad. Esto es más urgente que la brecha de facturación:
  sugiere que el proceso que debería mover una suscripción a `cancelled`/
  `expired` cuando vence no se está ejecutando, con el riesgo de seguir
  reportando como "ingreso recurrente activo" contratos que en la práctica
  ya terminaron.

### CRM

- **El pipeline abierto (prospect + qualification) suma ~\$45,9M, casi
  2,5 veces el monto ya ganado (\$18,4M)** — hay mucho valor represado en
  etapas tempranas. La tasa de cierre sobre lo ya decidido es saludable
  (61,1% ganado, 476 vs. 303 perdidas), pero el cuello de botella no está
  en cerrar bien lo que llega a negociación — está en que gran parte del
  pipeline nunca avanza más allá de `prospect`/`qualification`.
- **El canal con más volumen de leads es el que peor convierte**: `web`
  aporta 810 leads (el mayor volumen) pero solo 8,5% de conversión, contra
  14,7% de `cold_call` con apenas 204 leads. Antes de aumentar el
  presupuesto de adquisición web asumiendo que más volumen es
  proporcionalmente más ventas, conviene revisar la calificación de esos
  leads — puede que la vía más rentable a corto plazo sea reforzar
  `cold_call`/`referral`, no escalar el canal de mayor volumen.

### Cruce university ↔ billing y engagement CRM

- **El rendimiento académico no se relaciona con cuánto factura un
  estudiante-cliente**: alto rendimiento factura en promedio \$687,95,
  bajo rendimiento \$681,14 — prácticamente idéntico. Es un hallazgo
  negativo, pero útil: descarta una hipótesis razonable (que reprobar
  cursos esté asociado a menor gasto) antes de diseñar cualquier
  intervención — un programa de retención académica no debería
  justificarse por impacto en facturación, porque no lo hay.
- **Las cuentas CRM con al menos una oportunidad abierta reciben 43% más
  actividades de venta que las que no tienen ninguna** (4,08 vs. 2,85 en
  promedio) — el esfuerzo comercial se concentra donde ya hay una
  oportunidad en curso. Las cuentas "frías" (sin oportunidad, 2.738 de
  5.000) reciben menos de la mitad de la atención — son candidatas
  naturales para una campaña de prospección activa antes de asumir que no
  tienen potencial.